# Phase 2 — full time-series extraction

Extracts every block in `data/blocks.geojson`, April–October, 2019 → present,
into `data/processed/block_timeseries.parquet` matching the data contract:

`block_id, date, scene_id, ndvi_median, ndvi_p10, ndvi_p90, ndre_median, valid_frac`

All Earth Engine work is server-side in `vigor.extract` (one `reduceRegions`
per scene per resolution — 10 m NDVI stats chained into 20 m NDRE — no Python
loop over scenes, no `getInfo` in loops); results come back in a single table
`getInfo`. The same code path serves any number of blocks; if the table
outgrows `getInfo`, switch the pull to `extract.export_table_to_drive`.

**Acceptance criteria (Phase 2):**
- Curves show correct phenology: dormant 0.15–0.25, May green-up,
  July–August plateau, October decline.
- 2024 shows delayed green-up or depressed peaks exactly in the blocks with
  known January 2024 freeze damage — the most important validation in the project.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from vigor import extract, ingest, plots

RAW_CSV = PROJECT_ROOT / "data" / "raw" / "block_timeseries_raw.csv"
FIXTURE_CSV = PROJECT_ROOT / "tests" / "fixtures" / "sample_extract.csv"
PARQUET = PROJECT_ROOT / "data" / "processed" / "block_timeseries.parquet"
FIGURES = PROJECT_ROOT / "outputs" / "figures"

In [ ]:
extract.init_ee()

In [ ]:
blocks = extract.load_blocks(PROJECT_ROOT / "data" / "blocks.geojson")
blocks[["block_id", "site", "variety", "planting_year", "area_ha", "n_pixels"]]

## Extraction

Builds the lazy (scene × block) table server-side, then starts one Drive
export. This avoids Earth Engine's interactive aggregation limit for the
full 2019–present archive. Once the task is complete, rerun the notebook:
the CSV is downloaded automatically and all remaining cells run locally.

In [ ]:
if not RAW_CSV.exists():
    try:
        extract.download_latest_drive_csv("block_timeseries_raw.csv", RAW_CSV)
    except FileNotFoundError:
        table = extract.timeseries_table(blocks)
        task = extract.export_table_to_drive(table, description="block_timeseries_raw")
        print(f"started Drive export {task.id}; wait for COMPLETED, then rerun this notebook")
        raise RuntimeError("Extraction is running in Earth Engine. Rerun after the Drive task completes.")

raw = pd.read_csv(RAW_CSV)
print(f"using local raw table: {len(raw)} rows, {raw['date'].min()} to {raw['date'].max()}")

raw.head()

In [ ]:
RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
FIXTURE_CSV.parent.mkdir(parents=True, exist_ok=True)
raw.to_csv(RAW_CSV, index=False)
raw.to_csv(FIXTURE_CSV, index=False)
print(f"raw table -> {RAW_CSV}")
print(f"test fixture -> {FIXTURE_CSV}")

In [ ]:
ts = ingest.raw_to_timeseries(raw)
ingest.write_timeseries(ts, PARQUET)
print(f"parquet -> {PARQUET}\n")

summary = ts.groupby("block_id").agg(
    rows=("date", "size"),
    usable=("valid_frac", lambda s: int((s >= ingest.MIN_VALID_FRAC).sum())),
    first=("date", "min"),
    last=("date", "max"),
)
summary

## Multi-year NDVI curves

One panel per block, one line per year (light = oldest, dark = newest);
**2024, the freeze year, is drawn in red**. Only rows with
`valid_frac ≥ 0.8` are plotted.

Check: dormant-season shoulders 0.15–0.25, May green-up, July–August plateau,
October decline — and 2024 sitting visibly low or late where freeze damage is known.

In [ ]:
fig = plots.ndvi_multiyear(ts, emphasize_year=2024)
saved = plots.save_figure(fig, FIGURES / "02_all_ndvi_multiyear.png")
print(f"figure -> {saved}")

## Numeric freeze check

Median of the July block-median NDVI, per block per year — 2024 should stand
out low against each block's own history.

In [ ]:
usable = ingest.analysis_view(ts)
july = usable[usable["date"].dt.month == 7]
july.groupby(["block_id", july["date"].dt.year.rename("year")])["ndvi_median"].median().unstack().round(3)